In [ ]:
!pip install pennylane

$\renewcommand{\ket}[1]{|#1\rangle}$
# Training and evaluating quantum kernels

Kernel methods are one of the cornerstones of classical machine
learning. Here we are concerned with kernels that can be evaluated on
quantum computers, *quantum kernels* for short. In this tutorial you
will learn how to evaluate kernels, use them for classification and
train them with gradient-based optimization, and all that using the
functionality of PennyLane\'s [kernels
module](https://pennylane.readthedocs.io/en/latest/code/qml_kernels.html).
The demo is based on Ref., a project from Xanadu\'s own
[QHack](https://qhack.ai/) hackathon.

## What are kernel methods?

To understand what a kernel method does, let\'s first revisit one of the
simplest methods to assign binary labels to datapoints: linear
classification.

Imagine we want to discern two different classes of points that lie in
different corners of the plane. A linear classifier corresponds to
drawing a line and assigning different labels to the regions on opposing
sides of the line:

<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/linear_classification.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>

We can mathematically formalize this by assigning the label $y$ via

$$y(\boldsymbol{x}) = \operatorname{sgn}(\langle \boldsymbol{w}, \boldsymbol{x}\rangle + b).$$

The vector $\boldsymbol{w}$ points perpendicular to the line and thus
determine its slope. The independent term $b$ specifies the position on
the plane. In this form, linear classification can also be extended to
higher dimensional vectors $\boldsymbol{x},$ where a line does not
divide the entire space into two regions anymore. Instead one needs a
*hyperplane*. It is immediately clear that this method is not very
powerful, as datasets that are not separable by a hyperplane can\'t be
classified without error.

We can actually sneak around this limitation by performing a neat trick:
if we define some map $\phi(\boldsymbol{x})$ that *embeds* our
datapoints into a larger *feature space* and then perform linear
classification there, we could actually realise non-linear
classification in our original space!

<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/embedding_nonlinear_classification.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>

If we go back to the expression for our prediction and include the
embedding, we get

$$y(\boldsymbol{x}) = \operatorname{sgn}(\langle \boldsymbol{w}, \phi(\boldsymbol{x})\rangle + b).$$

We will forgo one tiny step, but it can be shown that for the purpose of
optimal classification, we can choose the vector defining the decision
boundary as a linear combination of the embedded datapoints
$\boldsymbol{w} = \sum_i \alpha_i \phi(\boldsymbol{x}_i).$ Putting this
into the formula yields

$$y(\boldsymbol{x}) = \operatorname{sgn}\left(\sum_i \alpha_i \langle \phi(\boldsymbol{x}_i), \phi(\boldsymbol{x})\rangle + b\right).$$

This rewriting might not seem useful at first, but notice the above
formula only contains inner products between vectors in the embedding
space:

$$k(\boldsymbol{x}_i, \boldsymbol{x}_j) = \langle \phi(\boldsymbol{x}_i), \phi(\boldsymbol{x}_j)\rangle.$$

We call this function the *kernel*. It provides the advantage that we
can often find an explicit formula for the kernel $k$ that makes it
superfluous to actually perform the (potentially expensive) embedding
$\phi.$ Consider for example the following embedding and the associated
kernel:

$$\begin{aligned}
\phi((x_1, x_2)) &= (x_1^2, \sqrt{2} x_1 x_2, x_2^2) \\
k(\boldsymbol{x}, \boldsymbol{y}) &= x_1^2 y_1^2 + 2 x_1 x_2 y_1 y_2 + x_2^2 y_2^2 = \langle \boldsymbol{x}, \boldsymbol{y} \rangle^2.
\end{aligned}$$

This means by just replacing the regular scalar product in our linear
classification with the map $k,$ we can actually express much more
intricate decision boundaries!

This is very important, because in many interesting cases the embedding
$\phi$ will be much costlier to compute than the kernel $k.$

In this demo, we will explore one particular kind of kernel that can be
realized on near-term quantum computers, namely *Quantum Embedding
Kernels (QEKs)*. These are kernels that arise from embedding data into
the space of quantum states. We formalize this by considering a
parameterised quantum circuit $U(\boldsymbol{x})$ that maps a datapoint
$\boldsymbol{x}$ to the state

$$|\psi(\boldsymbol{x})\rangle = U(\boldsymbol{x}) |0 \rangle.$$

The kernel value is then given by the *overlap* of the associated
embedded quantum states

$$k(\boldsymbol{x}_i, \boldsymbol{x}_j) = | \langle\psi(\boldsymbol{x}_i)|\psi(\boldsymbol{x}_j)\rangle|^2.$$


# A toy problem

In this demo, we will treat a toy problem that showcases the inner
workings of classification with quantum embedding kernels, training
variational embedding kernels and the available functionalities to do
both in PennyLane. We of course need to start with some imports:


In [ ]:
from pennylane import numpy as np
import matplotlib as mpl

np.random.seed(1359)

And we proceed right away to create a dataset to work with, the
`DoubleCake` dataset. Firstly, we define two functions to enable us to
generate the data. The details of these functions are not essential for
understanding the demo, so don\'t mind them if they are confusing.


In [ ]:
def _make_circular_data(num_sectors):
    """Generate datapoints arranged in an even circle."""
    center_indices = np.array(range(0, num_sectors))
    sector_angle = 2 * np.pi / num_sectors
    angles = (center_indices + 0.5) * sector_angle
    x = 0.7 * np.cos(angles)
    y = 0.7 * np.sin(angles)
    labels = 2 * np.remainder(np.floor_divide(angles, sector_angle), 2) - 1

    return x, y, labels


def make_double_cake_data(num_sectors):
    x1, y1, labels1 = _make_circular_data(num_sectors)
    x2, y2, labels2 = _make_circular_data(num_sectors)

    # x and y coordinates of the datapoints
    x = np.hstack([x1, 0.5 * x2])
    y = np.hstack([y1, 0.5 * y2])

    # Canonical form of dataset
    X = np.vstack([x, y]).T

    labels = np.hstack([labels1, -1 * labels2])

    # Canonical form of labels
    Y = labels.astype(int)

    return X, Y

Next, we define a function to help plot the `DoubleCake` data:


In [ ]:
def plot_double_cake_data(X, Y, ax, num_sectors=None):
    """Plot double cake data and corresponding sectors."""
    x, y = X.T
    cmap = mpl.colors.ListedColormap(["#FF0000", "#0000FF"])
    ax.scatter(x, y, c=Y, cmap=cmap, s=25, marker="s")

    if num_sectors is not None:
        sector_angle = 360 / num_sectors
        for i in range(num_sectors):
            color = ["#FF0000", "#0000FF"][(i % 2)]
            other_color = ["#FF0000", "#0000FF"][((i + 1) % 2)]
            ax.add_artist(
                mpl.patches.Wedge(
                    (0, 0),
                    1,
                    i * sector_angle,
                    (i + 1) * sector_angle,
                    lw=0,
                    color=color,
                    alpha=0.1,
                    width=0.5,
                )
            )
            ax.add_artist(
                mpl.patches.Wedge(
                    (0, 0),
                    0.5,
                    i * sector_angle,
                    (i + 1) * sector_angle,
                    lw=0,
                    color=other_color,
                    alpha=0.1,
                )
            )
            ax.set_xlim(-1, 1)

    ax.set_ylim(-1, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    return ax

Let\'s now have a look at our dataset. In our example, we will work with
3 sectors:


In [ ]:
import matplotlib.pyplot as plt

num_sectors = 3
X, Y = make_double_cake_data(num_sectors)

ax = plot_double_cake_data(X, Y, plt.gca(), num_sectors=num_sectors)

# Defining a Quantum Embedding Kernel

PennyLane\'s [kernels
module](https://pennylane.readthedocs.io/en/latest/code/qml_kernels.html)
allows for a particularly simple implementation of Quantum Embedding
Kernels. The first ingredient we need for this is an *ansatz*, which we
will construct by repeating a layer as building block. Let\'s start by
defining this layer:


In [ ]:
import pennylane as qp


def layer(x, params, wires, i0=0, inc=1):
    """Building block of the embedding ansatz"""
    i = i0
    for j, wire in enumerate(wires):
        qp.Hadamard(wires=[wire])
        qp.RZ(x[i % len(x)], wires=[wire])
        i += inc
        qp.RY(params[0, j], wires=[wire])

    n_wires = len(wires)
    for p, w in zip(params[1], wires):
        qp.CRZ(p, wires=[w % n_wires, (w + 1) % n_wires])

To construct the ansatz, this layer is repeated multiple times, reusing
the datapoint `x` but feeding different variational parameters `params`
into each of them. Together, the datapoint and the variational
parameters fully determine the embedding ansatz $U(\boldsymbol{x}).$ In
order to construct the full kernel circuit, we also require its adjoint
$U(\boldsymbol{x})^\dagger,$ which we can obtain via
[\`qp.adjoint]{.title-ref}.\`


In [ ]:
def ansatz(x, params, wires):
    """The embedding ansatz"""
    for j, layer_params in enumerate(params):
        layer(x, layer_params, wires, i0=j * len(wires))


adjoint_ansatz = qp.adjoint(ansatz)


def random_params(num_wires, num_layers):
    """Generate random variational parameters in the shape for the ansatz."""
    return np.random.uniform(0, 2 * np.pi, (num_layers, 2, num_wires), requires_grad=True)

Together with the ansatz we only need a device to run the quantum
circuit on. For the purpose of this tutorial we will use PennyLane\'s
`default.qubit` device with 5 wires in analytic mode.


In [ ]:
dev = qp.device("default.qubit", wires=5)
wires = dev.wires.tolist()

Let us now define the quantum circuit that realizes the kernel. We will
compute the overlap of the quantum states by first applying the
embedding of the first datapoint and then the adjoint of the embedding
of the second datapoint. We finally extract the probabilities of
observing each basis state.


In [ ]:
@qp.qnode(dev)
def kernel_circuit(x1, x2, params):
    ansatz(x1, params, wires=wires)
    adjoint_ansatz(x2, params, wires=wires)
    return qp.probs(wires=wires)

The kernel function itself is now obtained by looking at the probability
of observing the all-zero state at the end of the kernel circuit --
because of the ordering in `qp.probs`, this is the first entry:


In [ ]:
def kernel(x1, x2, params):
    return kernel_circuit(x1, x2, params)[0]

> > Note
>
> An alternative way to set up the kernel circuit in PennyLane would be
> to use the observable type
> [Projector](https://pennylane.readthedocs.io/en/latest/code/api/pennylane.Projector.html).
> This is shown in the [demo on kernel-based training of quantum
> models](https://pennylane.ai/qml/demos/tutorial_kernel_based_training),
> where you will also find more background information on the kernel
> circuit structure itself.

Before focusing on the kernel values we have to provide values for the
variational parameters. At this point we fix the number of layers in the
ansatz circuit to $6.$


In [ ]:
init_params = random_params(num_wires=5, num_layers=6)

Now we can have a look at the kernel value between the first and the
second datapoint:


In [ ]:
kernel_value = kernel(X[0], X[1], init_params)
print(f"The kernel value between the first and second datapoint is {kernel_value:.3f}")

The mutual kernel values between all elements of the dataset form the
*kernel matrix*. We can inspect it via the
`qp.kernels.square_kernel_matrix` method, which makes use of symmetry of
the kernel,
$k(\boldsymbol{x}_i,\boldsymbol{x}_j) = k(\boldsymbol{x}_j, \boldsymbol{x}_i).$
In addition, the option `assume_normalized_kernel=True` ensures that we
do not calculate the entries between the same datapoints, as we know
them to be 1 for our noiseless simulation. Overall this means that we
compute $\frac{1}{2}(N^2-N)$ kernel values for $N$ datapoints. To
include the variational parameters, we construct a `lambda` function
that fixes them to the values we sampled above.


In [ ]:
init_kernel = lambda x1, x2: kernel(x1, x2, init_params)
K_init = qp.kernels.square_kernel_matrix(X, init_kernel, assume_normalized_kernel=True)

with np.printoptions(precision=3, suppress=True):
    print(K_init)

# Using the Quantum Embedding Kernel for predictions

The quantum kernel alone can not be used to make predictions on a
dataset, becaues it is essentially just a tool to measure the similarity
between two datapoints. To perform an actual prediction we will make use
of scikit-learn\'s Support Vector Classifier (SVC).


In [ ]:
from sklearn.svm import SVC

To construct the SVM, we need to supply `sklearn.svm.SVC` with a
function that takes two sets of datapoints and returns the associated
kernel matrix. We can make use of the function
`qp.kernels.kernel_matrix` that provides this functionality. It expects
the kernel to not have additional parameters besides the datapoints,
which is why we again supply the variational parameters via the `lambda`
function from above. Once we have this, we can let scikit-learn adjust
the SVM from our Quantum Embedding Kernel.

> > Note
>
> This step does *not* modify the variational parameters in our circuit
> ansatz. What it does is solving a different optimization task for the
> $\alpha$ and $b$ vectors we introduced in the beginning.


In [ ]:
svm = SVC(kernel=lambda X1, X2: qp.kernels.kernel_matrix(X1, X2, init_kernel)).fit(X, Y)

To see how well our classifier performs we will measure which percentage
of the dataset it classifies correctly.


In [ ]:
def accuracy(classifier, X, Y_target):
    return 1 - np.count_nonzero(classifier.predict(X) - Y_target) / len(Y_target)


accuracy_init = accuracy(svm, X, Y)
print(f"The accuracy of the kernel with random parameters is {accuracy_init:.3f}")

We are also interested in seeing what the decision boundaries in this
classification look like. This could help us spotting overfitting issues
visually in more complex data sets. To this end we will introduce a
second helper method.


In [ ]:
def plot_decision_boundaries(classifier, ax, N_gridpoints=14):
    _xx, _yy = np.meshgrid(np.linspace(-1, 1, N_gridpoints), np.linspace(-1, 1, N_gridpoints))

    _zz = np.zeros_like(_xx)
    for idx in np.ndindex(*_xx.shape):
        _zz[idx] = classifier.predict(np.array([_xx[idx], _yy[idx]])[np.newaxis, :]).item()

    plot_data = {"_xx": _xx, "_yy": _yy, "_zz": _zz}
    ax.contourf(
        _xx,
        _yy,
        _zz,
        cmap=mpl.colors.ListedColormap(["#FF0000", "#0000FF"]),
        alpha=0.2,
        levels=[-1, 0, 1],
    )
    plot_double_cake_data(X, Y, ax)

    return plot_data

With that done, let\'s have a look at the decision boundaries for our
initial classifier:


In [ ]:
init_plot_data = plot_decision_boundaries(svm, plt.gca())

We see the outer points in the dataset can be correctly classified, but
we still struggle with the inner circle. But remember we have a circuit
with many free parameters! It is reasonable to believe we can give
values to those variational parameters which improve the overall
accuracy of our SVC.

# Training the Quantum Embedding Kernel

To be able to train the Quantum Embedding Kernel we need some measure of
how well it fits the dataset in question. Performing an exhaustive
search in parameter space is not a good solution because it is very
resource intensive, and since the accuracy is a discrete quantity we
would not be able to detect small improvements.

We can, however, resort to a more specialized measure, the
*kernel-target alignment*. The kernel-target alignment compares the
similarity predicted by the quantum kernel to the actual labels of the
training data. It is based on *kernel alignment*, a similiarity measure
between two kernels with given kernel matrices $K_1$ and $K_2:$

$$\operatorname{KA}(K_1, K_2) = \frac{\operatorname{Tr}(K_1 K_2)}{\sqrt{\operatorname{Tr}(K_1^2)\operatorname{Tr}(K_2^2)}}.$$

> > Note
>
> Seen from a more theoretical side, $\operatorname{KA}$ is nothing else
> than the cosine of the angle between the kernel matrices $K_1$ and
> $K_2$ if we see them as vectors in the space of matrices with the
> Hilbert-Schmidt (or Frobenius) scalar product
> $\langle A, B \rangle = \operatorname{Tr}(A^T B).$ This reinforces the
> geometric picture of how this measure relates to objects, namely two
> kernels, being aligned in a vector space.

The training data enters the picture by defining an *ideal* kernel
function that expresses the original labelling in the vector
$\boldsymbol{y}$ by assigning to two datapoints the product of the
corresponding labels:

$$k_{\boldsymbol{y}}(\boldsymbol{x}_i, \boldsymbol{x}_j) = y_i y_j.$$

The assigned kernel is thus $+1$ if both datapoints lie in the same
class and $-1$ otherwise and its kernel matrix is simply given by the
outer product $\boldsymbol{y}\boldsymbol{y}^T.$ The kernel-target
alignment is then defined as the kernel alignment of the kernel matrix
$K$ generated by the quantum kernel and
$\boldsymbol{y}\boldsymbol{y}^T:$

$$\operatorname{KTA}_{\boldsymbol{y}}(K)
= \frac{\operatorname{Tr}(K \boldsymbol{y}\boldsymbol{y}^T)}{\sqrt{\operatorname{Tr}(K^2)\operatorname{Tr}((\boldsymbol{y}\boldsymbol{y}^T)^2)}}
= \frac{\boldsymbol{y}^T K \boldsymbol{y}}{\sqrt{\operatorname{Tr}(K^2)} N}$$

where $N$ is the number of elements in $\boldsymbol{y},$ that is the
number of datapoints in the dataset.

In summary, the kernel-target alignment effectively captures how well
the kernel you chose reproduces the actual similarities of the data. It
does have one drawback, however: having a high kernel-target alignment
is only a necessary but not a sufficient condition for a good
performance of the kernel. This means having good alignment is
guaranteed for good performance, but optimal alignment will not always
bring optimal training accuracy with it.

Let\'s now come back to the actual implementation. PennyLane\'s
`kernels` module allows you to easily evaluate the kernel target
alignment:


In [ ]:
kta_init = qp.kernels.target_alignment(X, Y, init_kernel, assume_normalized_kernel=True)

print(f"The kernel-target alignment for our dataset and random parameters is {kta_init:.3f}")

Now let\'s code up an optimization loop and improve the kernel-target
alignment!

We will make use of regular gradient descent optimization. To speed up
the optimization we will not use the entire training set to compute
$\operatorname{KTA}$ but rather sample smaller subsets of the data at
each step, we choose $4$ datapoints at random. Remember that
PennyLane\'s built-in optimizer works to *minimize* the cost function
that is given to it, which is why we have to multiply the kernel target
alignment by $-1$ to actually *maximize* it in the process.

> > Note
>
> Currently, the function `qp.kernels.target_alignment` is not
> differentiable yet, making it unfit for gradient descent optimization.
> We therefore first define a differentiable version of this function.


In [ ]:
def target_alignment(
    X,
    Y,
    kernel,
    assume_normalized_kernel=False,
    rescale_class_labels=True,
):
    """Kernel-target alignment between kernel and labels."""

    K = qp.kernels.square_kernel_matrix(
        X,
        kernel,
        assume_normalized_kernel=assume_normalized_kernel,
    )

    if rescale_class_labels:
        nplus = np.count_nonzero(np.array(Y) == 1)
        nminus = len(Y) - nplus
        _Y = np.array([y / nplus if y == 1 else y / nminus for y in Y])
    else:
        _Y = np.array(Y)

    T = np.outer(_Y, _Y)
    inner_product = np.sum(K * T)
    norm = np.sqrt(np.sum(K * K) * np.sum(T * T))
    inner_product = inner_product / norm

    return inner_product


params = init_params
opt = qp.GradientDescentOptimizer(0.2)

for i in range(500):
    # Choose subset of datapoints to compute the KTA on.
    subset = np.random.choice(list(range(len(X))), 4)
    # Define the cost function for optimization
    cost = lambda _params: -target_alignment(
        X[subset],
        Y[subset],
        lambda x1, x2: kernel(x1, x2, _params),
        assume_normalized_kernel=True,
    )
    # Optimization step
    params = opt.step(cost, params)

    # Report the alignment on the full dataset every 50 steps.
    if (i + 1) % 50 == 0:
        current_alignment = target_alignment(
            X,
            Y,
            lambda x1, x2: kernel(x1, x2, params),
            assume_normalized_kernel=True,
        )
        print(f"Step {i+1} - Alignment = {current_alignment:.3f}")

We want to assess the impact of training the parameters of the quantum
kernel. Thus, let\'s build a second support vector classifier with the
trained kernel:


In [ ]:
# First create a kernel with the trained parameter baked into it.
trained_kernel = lambda x1, x2: kernel(x1, x2, params)

# Second create a kernel matrix function using the trained kernel.
trained_kernel_matrix = lambda X1, X2: qp.kernels.kernel_matrix(X1, X2, trained_kernel)

# Note that SVC expects the kernel argument to be a kernel matrix function.
svm_trained = SVC(kernel=trained_kernel_matrix).fit(X, Y)

We expect to see an accuracy improvement vs. the SVM with random
parameters:


In [ ]:
accuracy_trained = accuracy(svm_trained, X, Y)
print(f"The accuracy of a kernel with trained parameters is {accuracy_trained:.3f}")

We have now achieved perfect classification! 🎆

Following on the results that SVM\'s have proven good generalisation
behavior, it will be interesting to inspect the decision boundaries of
our classifier:


In [ ]:
trained_plot_data = plot_decision_boundaries(svm_trained, plt.gca())

Indeed, we see that now not only every data instance falls within the
correct class, but also that there are no strong artifacts that would
make us distrust the model. In this sense, our approach benefits from
both: on one hand it can adjust itself to the dataset, and on the other
hand is not expected to suffer from bad generalisation.


# Before you train: Pre-screening quantum kernels with geometric difference

Can we predict\-\--*before* investing a ton of research
hours\-\--whether a quantum kernel has the potential to beat a classical
one across all kernel methods?

From a practitioner's perspective, such a **pre-screening test** is
invaluable: it lets us rule out quantum kernels that don't offer any
potential quantum advantage right from the start.

Huang *et al.* introduced exactly this test. Their proposed **geometric
difference** $g$ metric is a single scalar that quantifies how
differently the geometries defined by two kernels represent your data.
The formula for $g$ is:

$$g = \sqrt{\|\sqrt{K_q} K_c^{-1} \sqrt{K_q}\|_\infty},$$

where $K_q$ and $K_c$ are quantum and classical Gram matrices,
respectively.

The following demonstration is designed to perform a pre-screening test
on several kernel methods---both classical and quantum---on a simple
classification task.

## What g tells us and why it is important

When $g \approx 1$, the quantum kernel's geometry is essentially the
same as a good classical kernel's. The quantum kernel offers no
geometric advantage, making it unlikely to outperform the classical
kernel **in any kernel-based learning algorithm** (e.g., SVM, Gaussian
Processes). Huang et al. proved this concept in a rigorous mathematical
way in their paper. Conversely, if $g >> 1$, the quantum geometry is
genuinely different. A kernel method using the quantum kernel *might*
offer an advantage.

The approach presented in this demo focuses on ruling out
underperforming quantum kernels before investing in training. From a
complexity theory point of view, computing $g$ scales as $O(n^3)$ due to
the matrix inversion, and the most expensive training algorithms such as
Gaussian Processes also scale as $O(n^3)$ so we might think we are not
saving any computational time. However, from a practical perspective,
the real savings come from avoiding wasted researcher effort.

When a quantum kernel performs poorly, researchers often spend days
exploring different algorithm hyperparameters, cross-validation
strategies, and implementation debugging. If $g \approx 1$, you
immediately know the quantum kernel's geometry offers no
advantage\-\--it's not your implementation, not your algorithm choice,
and not a hyperparameter issue. The kernel is fundamentally limited
compared to classical kernels on this specific dataset.

## Background context: kernels

A **kernel** is a function $k(x, x')$ that measures similarity between
data points without explicitly computing their feature representations
in high-dimensional spaces, thus lowering the computational cost.

An example of a classical kernel is the Radial Basis Function (RBF)
kernel given by $k(x, x') = \exp(-\gamma \|x - x'\|^2)$. It implicitly
computes the inner product $\langle\phi(x), \phi(x')\rangle$. The
feature map $\phi(x)$ projects to infinite dimensions, but it is never
calculated directly.

Quantum kernels are similar but leverage the Hilbert space of a quantum
computer. A quantum kernel is defined by
$k(x, x') = |\langle\psi(x)|\psi(x')\rangle|^2$, where $|\psi(x)\rangle$
is the quantum state encoding the classical data $x$. For $n$ qubits,
the quantum state lives in a $2^n$-dimensional Hilbert space that is
implicitly manipulated.

**Key concept**: The **kernel matrix** (Gram matrix) $K$ has entries
$K_{ij} = k(x_i, x_j)$ that store all pairwise similarities between data
points.

## Demonstration setup

This demonstration uses the synthetic two-moons dataset from
scikit-learn to perform a pre-screening test on four quantum kernel
variants against a classical RBF kernel. By calculating the geometric
difference and then training SVMs with each kernel, we illustrate how
this metric can filter out unpromising quantum kernels before investing
in extensive hyperparameter tuning and training. This procedure is
summarized as:

1.  **Dataset**: Synthetic two-moons data generated with `scikit-learn`.
2.  **Five kernels to compare**:
    -   **Classical baseline**: Gaussian-RBF kernel
    -   **Quantum kernels**:
        -   Separable-rotation embedding (E1)
        -   IQP-style embedding (E2)
        -   Projected kernel from E1 (maximizing $g$ as proposed by
            Huang et al.)
        -   Projected kernel from E2
3.  **Our approach**: Calculate $g$ values between the classical kernel
    and each quantum kernel.


In [ ]:
# We first start by generating and visualizing the artificial data
import numpy as np
import matplotlib.pyplot as plt
import scipy

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(422)

X_raw, y = make_moons(n_samples=300, noise=0.10, random_state=0)

# train/test split BEFORE any scaling (avoid data leakage) ------------
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.30, random_state=0, stratify=y
)

# standardize the data
scaler = StandardScaler().fit(X_train_raw)  # statistics from train only
X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Train size: {X_train.shape[0]}    Test size: {X_test.shape[0]}")

# visualize it using a scatter plot
plt.figure(figsize=(4, 4))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], s=15, alpha=0.8, label="class 0")
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], s=15, alpha=0.8, label="class 1")
plt.axis("equal")
plt.title("Two‑moons— training split (standardised)")
plt.legend(frameon=False)
plt.show()

# Quantum kernels: fidelity-based and projected variants

We consider **five different kernels** derived from three sources: a
classical RBF kernel and two quantum embedding circuits, **E1** and
**E2**. Each kernel defines a different geometry for measuring
similarity between data points.

The classical kernel is the **Radial basis function kernel (RBF).** This
classical baseline is defined as:

$$k_{\text{RBF}}(x, x') = \exp(-\gamma \|x - x'\|^2).$$

This maps data into an infinite-dimensional space where closer inputs
remain close, and distant ones become nearly orthogonal. It captures a
**geometric**, distance-based notion of similarity in input space.

The two quantum embedding circuits are:

-   **E1 -- Separable RX rotations.** Each input feature $x_j$ is
    encoded into a single qubit using an $RX(x_j)$ gate. The circuit is
    fully separable (no entanglement), producing the quantum state
    $\lvert \psi_{\text{E1}}(x) \rangle$.
-   **E2 -- IQP embedding.** PennyLane's `qp.IQPEmbedding` applies
    Hadamards, parameterized $RZ(x_j)$ rotations, and entangling ZZ
    gates. This creates an entangled quantum state
    $\lvert \psi_{\text{E2}}(x) \rangle$, inspired by Instantaneous
    Quantum Polynomial (IQP) circuits.

The four quantum kernels derived from the embedding circuits are defined
in the following way.

-   **QK -- Standard quantum kernels.** For both E1 and E2, the kernel
    is defined by the **fidelity** between quantum states:

    $$k_{\text{QK-E1}}(x, x') = |\langle \psi_{\text{E1}}(x) \mid \psi_{\text{E1}}(x') \rangle|^2,$$

    $$k_{\text{QK-E2}}(x, x') = |\langle \psi_{\text{E2}}(x) \mid \psi_{\text{E2}}(x') \rangle|^2,$$

    where $\psi_{\text{E1}}(x)$ and $\psi_{\text{E2}}(x)$ are the
    quantum states generated by E1 and E2 respectively. These kernels
    reflect how aligned two quantum feature states are in Hilbert space.

-   **PQK -- Projected quantum kernels (PQK-E1 / PQK-E2).** For a
    projected quantum kernel, instead of computing fidelity, the output
    quantum state $|\psi(x)\rangle$ is **measured** to extract the
    expectation values of Pauli operators:

    $$v(x) = \left[ \langle X_0 \rangle, \langle Y_0 \rangle, \langle Z_0 \rangle, \dots, \langle Z_{n-1} \rangle \right]$$

    A classical **RBF kernel** is then applied to these real-valued
    vectors:

    $$k_{\text{PQK}}(x, x') = \exp\left( -\gamma \| v(x) - v(x') \|^2 \right).$$

    We obtain two different projected quantum kernels from E1 and E1:

    $$k_{\text{PQK-E1}}(x, x') = \exp\left( -\gamma \|v_{\text{E1}}(x) - v_{\text{E1}}(x')\|^2 \right),$$

    $$k_{\text{PQK-E2}}(x, x') = \exp\left( -\gamma \|v_{\text{E2}}(x) - v_{\text{E2}}(x')\|^2 \right),$$

    where $v_{\text{E1 }}(x)$ and $v_{\text{E2}}(x)$ are the Pauli
    expectation vectors from E1 and E2, respectively.

Let\'s define the embedding circuits E1 and E2, and visualize them using
an auxiliary drawing function.


In [ ]:
import numpy as np
import pennylane as qp
import matplotlib.pyplot as plt

n_features = X_train.shape[1]
n_qubits = n_features


# -- E1: separable RX rotations ---------------------------------------------
def embedding_E1(features):
    for j, xj in enumerate(features):
        qp.RX(np.pi * xj, wires=j)


# -- E2: IQP embedding via PennyLane template --------------------------------
def embedding_E2(features):
    qp.IQPEmbedding(features, wires=range(n_features))


from utils import draw_circuits_side_by_side

draw_circuits_side_by_side(
    [embedding_E1, embedding_E2],
    ["E1 Embedding Circuit", "E2 Embedding Circuit"],n_qubits
)

# Gram matrix computation

Using the kernels defined above, we now build the **Gram (kernel)
matrices** required to compute the practitioner's metric $g$. For a
dataset of $N$ samples and a kernel function $k(\cdot, \cdot)$, the Gram
matrix $K \in \mathbb{R}^{N \times N}$ is defined entrywise as:

$$K_{ij} = k(x_i, x_j).$$

Each entry $K_{ij}$ measures how similar two data points are, and the
full matrix $K$ provides a **global view** of the data in the kernel's
feature space. We compute five such matrices, one for each kernel
defined above.

The Gram matrices will be used in downstream evaluations to compare
kernel geometries and analyze expressivity and generalization metrics
like $g$.

The following code builds all five Gram (kernel) matrices: Classical,
QK-E1, QK-E2, PQK-E1, and PQK-E2.


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

# ---------------------------------------------------------------------------#
# Classical RBF Gram matrix                                                  #
# ---------------------------------------------------------------------------#
def classical_rbf_kernel(X, gamma=1.0):
    return rbf_kernel(X, gamma=gamma)


K_classical = classical_rbf_kernel(X_train)
print(f"K_RBF shape: {K_classical.shape}")

# ---------------------------------------------------------------------------#
# Quantum fidelity-based Gram matrices                                       #
# ---------------------------------------------------------------------------#
dev = qp.device("default.qubit", wires=n_qubits, shots=None)


def overlap_prob(x, y, embed):
    """Probability of measuring |0…0⟩ after U(x) U†(y)."""

    @qp.qnode(dev)
    def circuit():
        embed(x)
        qp.adjoint(embed)(y)
        return qp.probs(wires=range(n_qubits))

    return circuit()[0]


def quantum_kernel_matrix(X, embed):
    return qp.kernels.kernel_matrix(X, X, lambda v1, v2: overlap_prob(v1, v2, embed))


print("Computing QK-E1 (fidelity)...")
K_quantum_E1 = quantum_kernel_matrix(X_train, embed=embedding_E1)

print("Computing QK-E2 (fidelity)...")
K_quantum_E2 = quantum_kernel_matrix(X_train, embed=embedding_E2)

print(f"K_QK_E1 shape: {K_quantum_E1.shape}")
print(f"K_QK_E2 shape: {K_quantum_E2.shape}")

Next, we compute the projected quantum kernels (Pauli vectors +
classical RBF).


In [ ]:
# ---------------------------------------------------------------------------#
# Projected quantum kernels                                                  #
# ---------------------------------------------------------------------------#
def get_pauli_vectors(embedding_func, X):
    """Returns Pauli expectation vectors for each input using the given embedding."""
    observables = []
    for i in range(n_qubits):
        observables.extend([qp.PauliX(i), qp.PauliY(i), qp.PauliZ(i)])

    @qp.qnode(dev)
    def pauli_qnode(x):
        embedding_func(x)
        return [qp.expval(obs) for obs in observables]

    vectors = [pauli_qnode(x) for x in X]
    return np.array(vectors)


def calculate_gamma(vectors):
    """Use heuristic gamma = 1 / (d * var) for RBF kernel on Pauli space."""
    d = vectors.shape[1]
    var = np.var(vectors)
    return 1.0 / (d * var) if var > 1e-8 else 1.0


def pqk_kernel_matrix(X, embedding_func):
    """Computes PQK kernel matrix from Pauli vectors + RBF kernel."""
    pauli_vecs = get_pauli_vectors(embedding_func, X)
    gamma = calculate_gamma(pauli_vecs)
    return rbf_kernel(pauli_vecs, gamma=gamma)


print("Computing PQK-E1 (Pauli + RBF)...")
K_pqk_E1 = pqk_kernel_matrix(X_train, embedding_E1)

print("Computing PQK-E2 (Pauli + RBF)...")
K_pqk_E2 = pqk_kernel_matrix(X_train, embedding_E2)

print(f"K_PQK_E1 shape: {K_pqk_E1.shape}")
print(f"K_PQK_E2 shape: {K_pqk_E2.shape}")

Let\'s now visualize the Gram Matrices. Each matrix shows how similar
data points are to each other: brighter colors indicate higher
similarity and different patterns indicate different geometries.


In [ ]:
import matplotlib.pyplot as plt

# Visualize first 20x20 subset of each Gram matrix for clarity
subset_size = 20
matrices = [K_classical, K_quantum_E1, K_quantum_E2, K_pqk_E1, K_pqk_E2]
titles = ["Classical RBF", "QK-E1", "QK-E2", "PQK-E1", "PQK-E2"]

rows, cols = 2, 3
fig, axes = plt.subplots(rows, cols, figsize=(14, 8))

# Flatten axes array to loop over
axes = axes.flatten()

for i, (K, title) in enumerate(zip(matrices, titles)):
    im = axes[i].imshow(K[:subset_size, :subset_size], cmap="viridis", aspect="equal")
    axes[i].set_title(title)
    axes[i].set_xlabel("Sample index")
    if i % cols == 0:  # only first column gets a y-label
        axes[i].set_ylabel("Sample index")
    plt.colorbar(im, ax=axes[i], fraction=0.046)

# Hide any unused subplots (since we have 5 matrices, but a 2x3 grid = 6 spots)
for j in range(len(matrices), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

We then compute the practitioner's metric $g$ for each quantum kernel,
according to the formula used in the paper by Huang et al.


In [ ]:
from scipy.linalg import sqrtm


def compute_g(K_classical, K_quantum, eps=1e-7):
    """
    Compute geometric difference g between classical and quantum kernels.
    Formula: g = sqrt( || sqrt(K_quantum) @ inv(K_classical) @ sqrt(K_quantum) || )
    """
    N = K_classical.shape[0]

    # Regularize and invert K_classical
    Kc_reg = K_classical + eps * np.eye(N)
    Kc_inv = np.linalg.inv(Kc_reg)

    # Compute the square root of the quantum kernel
    sqrt_Kq = sqrtm(K_quantum)
    # Construct M = sqrt(Kq) @ Kc⁻¹ @ sqrt(Kq)
    M = sqrt_Kq @ Kc_inv @ sqrt_Kq

    # g = sqrt(max eigenvalue of M)
    max_eigval = np.max(np.linalg.eigvalsh(M))
    return np.sqrt(np.maximum(max_eigval, 0.0))


# Compute g for all four quantum kernels
g_QK_E1 = compute_g(K_classical, K_quantum_E1)
g_QK_E2 = compute_g(K_classical, K_quantum_E2)
g_PQK_E1 = compute_g(K_classical, K_pqk_E1)
g_PQK_E2 = compute_g(K_classical, K_pqk_E2)

# Display results
print("\n--- Geometric Difference (g) ---")
print(f"g (RBF vs QK‑E1):    {g_QK_E1:.4f}")
print(f"g (RBF vs QK‑E2):    {g_QK_E2:.4f}")
print(f"g (RBF vs PQK‑E1):   {g_PQK_E1:.4f}")
print(f"g (RBF vs PQK‑E2):   {g_PQK_E2:.4f}")

# What does a high g mean?

We can see that in terms of $g$: PQK-E1 \> PQK-E2 \> QK-E1 \> QK-E2. A
common misconception is that a higher geometric difference $g$
automatically means better classification performance, which might lead
us to believe, for example, that in terms of final accuracy, the ranking
will also be PQK-E1 \> PQK-E2 \> QK-E1 \> QK-E2.

This intuition is understandable---after all, a larger $g$ suggests that
the quantum kernel perceives the data very differently from a classical
one. But as we'll see, **a higher** $g$ **doesn't always translate into
better accuracy, it just means there's higher potential for an
improvement over the classical model**. In fact, a higher $g$ can
sometimes correspond to worse performance on the original task.

Let's see this in action.


In [ ]:
# We train SVMs using each kernel and compare test accuracy
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel


def train_evaluate_svm(K_train, K_test, y_train, y_test, name):
    print(f"Training SVM with {name} kernel...")
    clf = SVC(kernel="precomputed")
    clf.fit(K_train, y_train)
    acc = accuracy_score(y_test, clf.predict(K_test))
    print(f"  Test accuracy: {acc:.4f}")
    return acc


results = {}

# Classical RBF
K_rbf_test = rbf_kernel(X_test, X_train)
results["Classical RBF"] = train_evaluate_svm(
    K_classical, K_rbf_test, y_train, y_test, "Classical RBF"
)

# Quantum Kernel E1
K_qk_e1_test = qp.kernels.kernel_matrix(
    X_test, X_train, lambda x, y: overlap_prob(x, y, embedding_E1)
)
results["QK-E1"] = train_evaluate_svm(K_quantum_E1, K_qk_e1_test, y_train, y_test, "Quantum E1")

# Quantum Kernel E2
K_qk_e2_test = qp.kernels.kernel_matrix(
    X_test, X_train, lambda x, y: overlap_prob(x, y, embedding_E2)
)
results["QK-E2"] = train_evaluate_svm(K_quantum_E2, K_qk_e2_test, y_train, y_test, "Quantum E2")

# PQK E1
pauli_test_E1 = get_pauli_vectors(embedding_E1, X_test)
gamma_E1 = calculate_gamma(np.vstack((get_pauli_vectors(embedding_E1, X_train), pauli_test_E1)))
K_pqk_e1_test = rbf_kernel(pauli_test_E1, get_pauli_vectors(embedding_E1, X_train), gamma=gamma_E1)
results["PQK-E1"] = train_evaluate_svm(K_pqk_E1, K_pqk_e1_test, y_train, y_test, "PQK E1")

# PQK E2
pauli_test_E2 = get_pauli_vectors(embedding_E2, X_test)
gamma_E2 = calculate_gamma(np.vstack((get_pauli_vectors(embedding_E2, X_train), pauli_test_E2)))
K_pqk_e2_test = rbf_kernel(pauli_test_E2, get_pauli_vectors(embedding_E2, X_train), gamma=gamma_E2)
results["PQK-E2"] = train_evaluate_svm(K_pqk_E2, K_pqk_e2_test, y_train, y_test, "PQK E2")

# Summary
print("\n--- Accuracy Comparison ---")
for model, acc in results.items():
    print(f"{model:>15}: {acc:.4f}")

# Accuracy Comparison

import matplotlib.pyplot as plt

# Extract model names and accuracies
model_names = list(results.keys())
accuracies = [results[name] for name in model_names]

# Create bar chart
plt.figure(figsize=(10, 4))
bars = plt.bar(model_names, accuracies)
plt.ylim(0, 1.10)
plt.ylabel("Test Accuracy")
plt.title("SVM Accuracy by Kernel")
plt.xticks(rotation=15)
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Annotate values
for bar, acc in zip(bars, accuracies):
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2, yval + 0.015, f"{acc:.2f}", ha="center", va="bottom"
    )

plt.show()

Our test results reveal an important subtlety: **A higher geometric
difference** $g$ **does not guarantee better classification accuracy.**
For instance, **PQK‑E2** achieved perfect test accuracy ($100\%$),
despite having a lower $g$ than PQK‑E1.

This highlights a key message from the paper: The role of $g$ is *not*
to predict which kernel will perform best on a given task, but rather to
obtain a collection of kernels that have the *potential* to offer an
advantage.

Here, PQK-E1 and PQK-E2 both had the potential for an advantage over
classical, but PQK-E2 is the only one that actually achieved the
advantage. As a simple practical rule, if $g$ is low, then we can
immediately discard a quantum kernel, whereas if $g$ is high, we keep
the kernel as a potential solution because it offers a potential for an
improvement on our classification problem. This way, we have an
important diagnostic tool to filter out bad quantum kernels for our
data.

# Conclusion

In this notebook, we explored a fundamental question in quantum machine
learning: Can we anticipate, before training, whether a quantum kernel
might outperform a classical one?

To address this, we used the geometric difference $g$, a pre-training
metric introduced by Huang et al. that quantifies how *differently* a
quantum kernel organizes the data compared to a classical kernel. The
main takeaways from this demonstration are:

-   $g$ **is a diagnostic, not a performance predictor.** A large $g$
    indicates that the quantum kernel induces a very different geometry
    from the classical one---a *necessary*, but not *sufficient*,
    condition for quantum advantage.
-   **Higher** $g$ **does not imply higher accuracy.** In our results,
    **PQK‑E2** had a high $g$ and achieved perfect accuracy, but
    **PQK‑E1**, with a higher $g$, obtained a lower accuracy on the
    original task. This confirms that $g$ measures *potential*, not
    realized performance.
-   $g$'s **value is in ruling out unpromising kernels.** Kernels with
    very small $g$ are unlikely to offer any meaningful advantage over
    classical methods---the quantum kernel introduces no genuinely new
    distinctions beyond what a classical RBF can produce. By contrast, a
    high $g$ only tells us that *some advantage may be possible*, not
    that it will be realized.

What if we took labels into account? The authors in[^1] proposed a
method to artificially construct new labels that align with a quantum
kernel's geometry. This is a toy construction, but pretty fun to play
around with. Feel free to do so!

[^1]: Huang, Hsin-Yuan, Michael Broughton, Masoud Mohseni, Ryan Babbush,
    Sergio Boixo, Hartmut Neven, and Jarrod R. McClean. \"Power of data
    in quantum machine learning.\" [Nature Communications 12, 2631
    (2021)](https://www.nature.com/articles/s41467-021-22539-9).


In [ ]:
%matplotlib inline

# Seeing Phase Transitions with Quantum Computers

## Introduction

A phase transition occurs when a system undergoes an abrupt, qualitative
change in one or more of its properties\-\--for instance, liquid water
freezing to form ice. Phase transitions are of significant importance
across many areas of physics, including condensed matter physics,
cosmology, and high-energy physics.

They are important for several reasons: they facilitate the discovery of
new quantum states of matter, help illuminate entanglement and
long-range correlations within quantum systems, and help us understand
the behaviour of many different quantum systems at the same time (due to
the property of universality).

This tutorial introduces three quantum phase transitions relevant to
condensed matter physics and demonstrates how to simulate them on a
quantum computer. The transitions covered here include the
one-dimensional (1D) and two-dimensional (2D) quantum Ising model and a
dynamical quantum phase transition (i.e., phase transitions occurring in
time evolution).

> Note
>
> *Quantum* phase transitions are fundamentally different from
> *classical* phase transitions. While classical transitions are driven
> by thermal fluctuations, quantum phase transitions always occur at
> zero temperature and are induced by quantum fluctuations (i.e.,
> Heisenberg\'s uncertainty principle).

```{=html}
<br>
```
Studying these transitions analytically is challenging; the associated
discontinuities can cause mathematical models to break down. Although
classical numerical simulations have been widely used, the required
computational resources for certain cases can be prohibitive. But
there\'s another way to study phase transitions: using a quantum
computer. Potentially, they can compute aspects of phase transitions
more efficiently than any conventional technique.

To date, quantum computers have been employed to investigate quantum
phase transitions in diverse areas, including those related to the early
universe and high-energy particle colliders, a topological transition in
an Ising-like model, the transverse Ising model, and noisy quantum
systems. Furthermore, these systems have been applied to studying scalar
quantum field theory and the evolution of the universe.

Note: This tutorial focuses on the *quantum* Ising model. It complements
existing content on this model: [3-qubit Ising model in
PyTorch](https://pennylane.ai/qml/demos/tutorial_isingmodel_PyTorch),
[Transverse-field Ising
model](https://pennylane.ai/datasets/transverse-field-ising-model), and
[Quadratic Unconstrained Binary Optimization
(QUBO)](https://pennylane.ai/qml/demos/tutorial_QUBO)

## What is the Ising model?

The simplest Ising model consists of $N$ qubits arranged along a line.


<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/Fig_1_Ising_chain.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>


Each qubit interacts with the qubits on either side of it. For example,
the second qubit interacts with the first and third qubits.


The system's Hamiltonian is

$$\begin{equation}
H = -J \,\, \Sigma_{i=0}^{N-2} \sigma_{z}^{(i)} \sigma^{(i+1)}_{z},
\end{equation}$$

where $\sigma_{z}^{(i)}$ is the Pauli-Z operator for the $i^{th}$ qubit
and $J$ is the interaction strength between neighbouring qubits.

The code below creates this Hamiltonian for three qubits:


In [ ]:
import pennylane as qp

from pennylane import numpy as np

N = 3
J = 2
wires = range(N)

dev = qp.device("lightning.qubit", wires=N)

coeffs = [-J] * (N - 1)

obs = []
for i in range(N - 1):
    obs.append(qp.Z(i) @ qp.Z(i + 1))
H = qp.Hamiltonian(coeffs, obs)

print(f"H={H}")

# Why is the Ising Model Important?

At first glance, the Ising model looks like it\'s simple and
unrealistic. However, it correctly models many properties of real-world
magnets. Also, its simplicity allows us to actually solve it. You can
think of the Ising model as a sandbox to play in and quickly learn about
the essence of various complex real-world phenomena.

The Ising model exhibits a wide range of interesting emergent
properties, such as phase transitions. One calculation that gives us
insight into their behaviour is finding the ground state of the Ising
model and seeing how it changes as the interactions change. Often,
we\'re looking to see if a phase transition happens.

Let\'s look at an example.

# Seeing Phase Transitions with Quantum Computers

To do this, we\'ll use the well-known variational quantum eigensolver
(VQE) algorithm to find the ground state. You can find an introduction
to it [here](https://pennylane.ai/qml/demos/tutorial_vqe).

Let\'s start by finding the ground state of the Ising model for a fixed
value of $J$. We\'ll use the Hardware Efficient Ansatz (HEA) to do this.
It\'s a general-purpose ansatz that efficiently represents a wide range
of quantum states, it consists of:

1.  Applying three single-qubit rotations to each qubit. Each one is
    parameterized by a different rotation angle.
2.  Applying a CNOT gate to each neighbouring pair of qubits.
3.  Applying three single-qubit rotations to each qubit. Again, each one
    is parameterized by a different rotation angle.


In [ ]:
import random

random.seed(a=10)

# params is an array that stores the parameter values of the statevector that we use in VQE.
# Generate some initial random angle values.
params = np.array([2 * np.pi * random.uniform(0, 1)] * (6 * N), requires_grad=True)

# create an ansatz using the hardware efficiency ansatz (HEA)
def create_ansatz(params, N):
    # STEP 1: perform single-qubit rotations on all the qubits
    for i in range(N):
        qp.RZ(phi=params[i], wires=i)
        qp.RX(phi=params[N + i], wires=i)
        qp.RZ(phi=params[2 * N + i], wires=i)
    
    # STEP 2: perform a CNOT gate on each pair of neighbouring qubits
    for i in range(N - 1):
        qp.CNOT(wires=[i, i + 1])

    # STEP 3: perform single-qubit rotations on all the qubits
    for i in range(N):
        qp.RZ(phi=params[3 * N + i], wires=i)
        qp.RX(phi=params[4 * N + i], wires=i)
        qp.RZ(phi=params[5 * N + i], wires=i)

@qp.qnode(dev)
def quantum_circuit(params):
    # Create a quantum state using params
    create_ansatz(params, N)
    return qp.expval(H)

max_iters = 200
tolerance = 1e-04

# create an optimizer
opt = qp.GradientDescentOptimizer(stepsize=0.1)

# energy is a list that stores all the estimates for the ground-state energy
energy = []

# execute the VQE optimization loop
for i in range(max_iters):
    params, prev_energy = opt.step_and_cost(quantum_circuit, params)
    energy.append(prev_energy)

    if i > 1:
        if np.abs(energy[-2] - energy[-1]) < tolerance:
            break

# graph the energy as a function of the number of iterations
import matplotlib.pyplot as plt

plt.plot(list(range(len(energy))), energy)
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.show()

The graph above shows that the energy $E$ gradually decreases until it
reaches $E = - 4$. To check that this result makes sense, let\'s think
about the Hamiltonian. Consider the first term,
$-2 \sigma_{z}^{(0)} \sigma_{z}^{(1)}$. When the first two qubits are in
the computational basis state $| 0 \rangle$ , the product
$\langle \sigma_{z}^{(0)} \sigma_{z}^{(1)} \rangle$ is $(+1)(+1) = +1$.
Multiplying this by $J = -2$ gives an energy of -2. When the last two
qubits are in the state $| 0 \rangle$, the second term
$-2 \langle \sigma_{z}^{(1)} \sigma_{z}^{(2)} \rangle$ is also $E = -2$.
Combining these results gives $E = -2 -2 = -4$. When all the qubits are
in the other basis state $(| 1 \rangle)$, we also get $E = -4$. These
two calculations agree with the numerical result from VQE. So far, so
good.

Let\'s now introduce an extra energy term that\'s proportional to the
sum of all the Pauli X operators:

$$- h_{x}\Sigma_{i=0}^{N-1} \sigma_{x}^{(i)}.$$

If our qubits are actually spin-1/2 particles (e.g., electrons), $h_{x}$
is a horizontal magnetic field. Often, it\'s called a *transverse
field*. We introduce the extra term to increase the system\'s complexity
and to see if it leads to any interesting phase transitions. As you\'ll
see, it does.


<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/Fig_2_transverse_Ising.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>


The system\'s Hamiltonian becomes

$$H = -J \,\, \Sigma_{i=0}^{N-2} \sigma_{z}^{(i)} \sigma^{(i+1)}_{z} - h_{x}\Sigma_{i=0}^{N-1} \sigma_{x}^{(i)}.$$

A quantum phase transition happens when we change the ratio $J/h_x$.
Physically, this corresponds to changing the relative strength of the
coupling interaction and the horizontal magnetic field. When $J$ is much
larger than $h_{x}$, the ground state corresponds to all the spins
(i.e., the qubits) being aligned vertically (parallel to the $z$ axis).


<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/Fig_3_ground_state_J_large.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>


But, when $h_{x}$ is much greater than $J$, the ground state corresponds
to all the spins being aligned along the $x$ axis parallel to the
magnetic field:


<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/Fig_4_ground_state_h_large.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>


When $J/h_{x} = 1$, the ground state suddenly switches from the first
state (all vertical) to the second one (all horizontal). This is a
quantum phase transition. The interplay between $J$ and $h_{x}$ is like
a tug of war. The coupling constant $J$ tries to align all the qubits
vertically, in the computational basis. The magnetic field $h_{x}$ tries
to align them horizontally, in the Pauli X basis. Depending on value of
$J/h_{x}$, one of the two constants will dominate.

To see the phase transition, let\'s introduce the total magnetization
operator $M$ of all the qubits:

$$M =\frac{1}{N} \Sigma_{i} \sigma_{Z}^{(i)}.$$

It\'s just the sum of all the Pauli $Z$ operators scaled by the number
of qubits. For example, for the state
$| \psi \rangle = |0 \rangle |0\rangle$,
$\langle M \rangle = \frac{1}{2} \left( 1 + 1 \right) = 1$. The total
magnetization tracks the phase change as follows:

-   When $h_{x} \gg  J$, $\langle M \rangle = 0$ as each qubit is in an
    equal superposition of $|0 \rangle$ and $|1 \rangle$.
-   When $J \gg h_{x}$, $|  \langle M \rangle | = 1$ as the qubits are
    either all in $|0 \rangle$ or all in $|1 \rangle$.


Let\'s now calculate $\langle M \rangle$ for a range of $J/h_{x}$
values.


In [ ]:
N = 5
wires = range(N)

# h_x is the strength of the transverse magnetic field
h_x = 1

# Vary the value of the coupling constant J (change J/h_x)
J_list = [0.0, 0.25, 0.75, 0.9, 1.0, 1.1, 2.0, 5.0, 7.5]

# Store the expectation values of the magnetization operator M for different values of J/h_x
magnetization_list = []

dev_2 = qp.device("lightning.qubit", wires=N)

# This function prepares an estimate of the ground state & calculates its energy.
@qp.qnode(dev_2)
def quantum_circuit_2(params):
    # Generate an estimate of the ground state
    create_ansatz(params, N)
    return qp.expval(H)

# A function that returns the magnetization operator of N qubits.
def magnetization_op(N):
    total_op = qp.PauliZ(0)

    if N > 1:
        for i in range(1, N):
            total_op = total_op + qp.PauliZ(i)

    return total_op / N

#Prepare a parameterized state & return the value of the magnetization operator.
@qp.qnode(dev_2)
def calculate_magnetization(params):
    create_ansatz(params, N)
    return qp.expval(magnetization_op(N))

# Loop through all the different values of J
for i in range(len(J_list)):

    # Build the Hamiltonian

    # Add Pauli Z-Pauli Z interaction terms to the Hamiltonian
    coeffs = [-J_list[i]] * (N - 1)

    obs = []
    for j in range(N - 1):
        obs.append(qp.Z(j) @ qp.Z(j + 1))

    # Add Pauli X terms to the Hamiltonian
    for j in range(N):
        obs.append(qp.X(j))
        coeffs.append(-h_x)

    H = qp.Hamiltonian(coeffs, obs)

    params = np.array([2 * np.pi * random.uniform(0, 1)] * (6 * N), requires_grad=True)

    max_iters = 200
    tolerance = 1e-04

    # create an optimizer
    opt = qp.MomentumOptimizer(stepsize=0.02, momentum=0.9)

    energy = []

    # Run the VQE optimization loop
    for j in range(max_iters):
        params, prev_energy = opt.step_and_cost(quantum_circuit_2, params)
        energy.append(prev_energy)

        if j > 1:
            if np.abs(energy[-2] - energy[-1]) < tolerance:
                break

    magnetization_list.append(calculate_magnetization(params))

Now that we\'ve calculated $\langle M \rangle$, let\'s plot the results.


In [ ]:
# Plot |<magnetization>| versus J
plt.plot(J_list, np.abs(magnetization_list), marker="x")
plt.xlabel("J")
plt.ylabel(r"$|\langle M \rangle|$")
plt.title(r"$| \langle M \rangle |$ vs. $J$ for $N$=" + str(N))
plt.show()

Notice how the magnetization increases sharply around $J/h_{x} = 1$.
This suggests that a phase transition is happening. (It\'s also well
known that a phase transition does happen at this value.) Why doesn\'t
the graph have a sharp and discontinuous increase at exactly $J/h_x=1$?
There are two reasons:

-   Like all other numerical results, this result is just approximate.
-   The phase transition happens at $J/h_x=1$ in the asymptotic limit of
    large $N$, i.e., as the number of qubits goes to infinity. You can
    see this by plotting how $M$ changes for three different values of
    $N = 4, 5, 6$.


In [ ]:
# magnetization values for N = 4
magnetization_4 = [
    0.01705303,
    -0.05617393,
    0.34882499,
    0.38068118,
    0.74856645,
    0.90577316,
    0.9872206,
]
J_list_4 = [0.0, 0.25, 0.75, 0.9, 1.1, 2.0, 5.0]

# magnetization values for N = 6
magnetization_6 = [
    -0.11958867,
    0.00284093,
    0.01237123,
    0.00255386,
    0.81125517,
    0.92437233,
    0.99013448,
]
J_list_6 = J_list_4[:]

# Plot |<M>| for multiple N values versus J
plt.plot(J_list_4, np.abs(magnetization_4), "xk-", label="N=4")
plt.plot(J_list[0:8], np.abs(magnetization_list[0:8]), "xb--", label="N=5")
plt.plot(J_list_6, np.abs(magnetization_6), "sg:", label="N=6")

plt.xlabel("J")
plt.ylabel(r"$|\langle M \rangle|$")
plt.title(r"$| \langle M \rangle |$ vs. $J$ for $N$=4, 5, 6")
plt.legend(loc="lower right")
plt.show()

Notice how the increase in $|\langle M \rangle |$ gets steeper as we
increase $N$. You can think of this as showing that we\'re getting
closer and closer to the asymptotic behaviour of a truly discontinuous
phase transition.


# Two-dimensional Ising Model

In the 2D quantum Ising model, the qubits are arranged in a 2D grid.


<figure>
<img src='https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-9-images/Fig_5_2D_Ising_model.png?raw=1' alt='' style='display:block; margin:auto; width:50%;'/>
</figure>


Compared to the 1D model, the 2D model is richer, harder to solve
mathematically, and harder to simulate on classical computers. It\'s
also more realistic and is used by physicists to study low-dimensional
quantum systems. In this section, we\'ll explore phase transitions in
the 2D quantum Ising model. The Hamiltonian for the model is

$$H = -J \,\, \Sigma_{\langle i,j \rangle} \sigma_{z}^{(i)} \sigma^{(j)}_{z} - h_{x} \Sigma_{ i } \sigma_{x}^{(i)}$$

The expression $\langle i,j \rangle$ includes every pair of neighbouring
qubits in the lattice. The $\Sigma_{i}$ term sums over every qubit in
the lattice. The code below creates the Hamiltonian using [PennyLane\'s
spin
module](https://docs.pennylane.ai/en/stable/code/api/pennylane.spin.transverse_ising.html).


In [ ]:
N = 2

H = qp.spin.transverse_ising(lattice="square", n_cells=[N, N], h=1.0, boundary_condition=True)

print(f"H={H}")

Like we did for the 1D model, let\'s find the ground state using VQE.


In [ ]:
wires_2D = range(N**2)
dev_2D = qp.device("lightning.qubit", wires=wires_2D)

random.seed(a=10)

# generate random parameter values for the initial statevector
params = np.array([2 * np.pi * random.uniform(0, 1)] * (6 * N), requires_grad=True)

@qp.qnode(dev_2D)
def quantum_circuit_2D(params):
    create_ansatz(params, N)
    return qp.expval(H)

max_iters = 500
tolerance = 3e-04

# create an optimizer
opt = qp.GradientDescentOptimizer(stepsize=0.015)

energy = []

# execute the optimization loop
for i in range(max_iters):
    params, prev_energy = opt.step_and_cost(quantum_circuit_2D, params)
    energy.append(prev_energy)

    if i > 1:
        if np.abs(energy[-2] - energy[-1]) < tolerance:
            break

# print out the results
plt.plot(list(range(len(energy))), energy)
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.show()

The energy in the graph approaches -4.5, which makes sense. The
Hamiltonian has four two-qubit interaction terms and the smallest that
each one can be is -1. So, the ground-state energy must be less than -4.
Let\'s vary the ratio $J/h_{x}$ again and calculate the average value of
the magnetization $\langle M \rangle$ each time. Finally, let\'s plot
the results and see if there\'s a quantum phase transition.


In [ ]:
N = 3
dev_2D_varying_J = qp.device("lightning.qubit", wires=N**2)

# strength of transverse magnetic field
h_x = 1

# Vary J in order to see a phase transition in the magnetization as we change J/h_x
J_list = [0.035, 0.05, 0.1, 0.25, 0.375, 0.5, 0.75, 1.0, 5, 10]

magnetization_list = []

# Prepare a parameterized state & calculate the value of the magnetization operator.
@qp.qnode(dev_2D_varying_J)
def calculate_magnetization_2D(params):
    create_ansatz(params, N)
    return qp.expval(magnetization_op(N))

@qp.qnode(dev_2D_varying_J)
def quantum_circuit_2D_varying_J(params):
    create_ansatz(params, N)
    return qp.expval(H)

# Loop through all values of J
for i in range(len(J_list)):
    H = qp.spin.transverse_ising(
        lattice="square", coupling=J_list[i], n_cells=[N, N], boundary_condition=True
    )
    #Set the initial values of the rotation angle parameters.
    #The values below were chosen through trial and error.
    params = np.zeros(6 * N, requires_grad=True)
    for i in range(N):
        params[i] = 0
        params[N + i] = np.pi / 2
        params[2 * N + i] = np.pi / 2

    max_iters = 500

    # create an optimizer
    opt = qp.MomentumOptimizer(stepsize=0.03, momentum=0.9)

    energy = []

    # execute the optimization loop
    for j in range(max_iters):
        params, prev_energy = opt.step_and_cost(quantum_circuit_2D_varying_J, params)
        energy.append(prev_energy)

        if j > 1:
            if np.abs(energy[-2] - energy[-1]) < tolerance:
                break

    magnetization_list.append(calculate_magnetization_2D(params))

Let\'s plot the results.


In [ ]:
# Plot |<magnetization>| versus J
plt.plot(J_list, np.abs(magnetization_list), marker="x")
plt.xlabel("J")
plt.ylabel(r"$| \langle M \rangle |$")
plt.title(r"$| \langle M \rangle |$ vs. $J$ for $N$=" + str(N))
plt.show()

From the graph, it\'s unclear if there\'s a phase transition. Looking at
it, multiple data points are bunched up on the left. To spread them out,
let\'s change the scale by ignoring the last two points.


In [ ]:
plt.plot(J_list[0:8], np.abs(magnetization_list[0:8]), marker="x")
plt.xlabel("J")
plt.ylabel(r"$| \langle M \rangle |$")
plt.title(r"$| \langle M \rangle |$ vs. $J$ for $N$=" + str(N))
plt.show()

Like in the 1D case, the magnetization displays a rapid increase. This
is consistent with a phase change but it\'s not conclusive as the
increase is somewhat gradual. This is because $N$ is so small. Note that
the result is consistent with where the phase change is known to occur,.

# Time Evolution & Dynamical Phase Transitions

Another important aspect of quantum systems is how they evolve over
time. Sometimes, this evolution is hard to simulate on classical
computers. So, researchers are interested in modelling it on quantum
computers. Occasionally, some property of a quantum system changes
abruptly. This is called a *dynamical quantum phase transition*: a phase
transition that happens during the time evolution of a quantum system.

To evolve the Ising model in time, we\'ll use the well-known
Suzuki-Trotter product approximation. The code below does this.


In [ ]:
import math

N = 5
wires = range(N)

# Create H for a 1D Ising model with transverse & longitudinal magnetic fields
# We do this to copy what was done in Reference 13: https://arxiv.org/abs/2008.04894
obs = []
for j in range(N - 1):
    obs.append(qp.Z(j) @ qp.Z(j + 1))
obs.append(qp.Z(N-1) @ qp.Z(0))

# add Pauli X terms to Hamiltonian (transverse field)
for j in range(N):
    obs.append(qp.X(j))

# add Pauli Z terms to Hamiltonian (longitudinal field)
for j in range(N):
    obs.append(qp.Z(j))

dev = qp.device("lightning.qubit", wires=N)

J = -0.1

# strength of transverse field interaction
h_x = 1

# strength of longitudinal field interaction
h_z = -0.15

J_coeffs = [-J] * N

X_coeffs = [h_x] * N

Z_coeffs = [h_z] * N

coeffs = J_coeffs + X_coeffs + Z_coeffs

H = qp.Hamiltonian(coeffs, obs)

# create the circuit that evolves the system in time
@qp.qnode(dev)
def time_evolution_circuit(H, T):
    #Evolve the system via a sequence of short approximate Trotter time steps
    #https://docs.pennylane.ai/en/stable/code/api/pennylane.TrotterProduct.html
    qp.TrotterProduct(H, time=T, n=math.ceil(T / 0.1)+1, order=2)

    # return the final probabilities
    return qp.probs(wires=range(N))

To see if a dynamical phase transition happens, let\'s consider an
observable called the *rate function* $\gamma$. It depends on the
overlap between the quantum state that we start with and the final state
at some time $t$. More specifically,

$$\gamma = -\frac{1}{N} \log_{e} (|G|^{2}),$$

where $G = \langle \psi_{i} | \psi_{f}\rangle$, and $| \psi_{i}\rangle$
and $| \psi_{f} \rangle$ are the initial and final states, respectively.
As the system evolves, we\'ll keep calculating $\gamma$. If it changes
discontinuously, then a dynamical phase transition has happened.

The function below calculates $\gamma$ at time $T$.


In [ ]:
def rate_function(H, T, N):
    probability_list = time_evolution_circuit(H, T)
    mag_G_squared = probability_list[0]
    return -1 / N * np.log(mag_G_squared)

Let\'s now calculate $\gamma$ at different times to see how it evolves.
Finally, let\'s graph the value of $\gamma$ versus time to see if a
dynamical phase transition happens.


In [ ]:
rate_function_list = []

# time step size for time evolution
deltaT = 0.05

num_time_steps = 50

for i in range(num_time_steps):
    rate_function_list.append(rate_function(H, i * deltaT, N))

plt.plot(np.linspace(0, deltaT * (num_time_steps-1), num_time_steps), rate_function_list)
plt.xlabel("time")
plt.ylabel(r"Rate function, $\gamma$")
plt.title("Rate Function versus time")
plt.legend(["N=" + str(N)])
plt.show()

Notice the discontinuous change at $t = 1.5$. There, the derivative
$\frac{d \gamma}{d t}$ is undefined. Phase transitions are characterized
by discontinuous changes. So, the discontinuity suggests that a phase
change is happening. This conclusion is supported by classical numerical
simulations that show a phase transition at $t = 1.5$[^1].


[^1]: Stefano De Nicola, Alexios A. Michailidis, and Maksym Serbyn.
    \"Entanglement View of Dynamical Quantum Phase Transitions\", Phys.
    Rev. Lett. 126 040602 (2021), Figure 1 (d)
